In [1]:
import pandas as pd

# Datas pt-BR
import locale
locale.setlocale(locale.LC_TIME, 'pt_BR.UTF-8')


'pt_BR.UTF-8'

## <font color="#384572"> Importação e Contexto dos Dados</font>
Os dados do presente notebook foram extraídos do site da __Pós-Graduação em Física__ da __UFMG__ que cobre o Corpo Discente do Programa. 

<font color="#cb1027"> Fonte: </font>[Física UFMG = Corpo Discente do 2º semestre de 2025]("https://www.fisica.ufmg.br/posgraduacao/corpo-discente/")

In [2]:
# Source of data: 
source_data: str = "https://www.fisica.ufmg.br/posgraduacao/corpo-discente/"

# Loading data
mest, doc = pd.read_html(source_data, header=1)
mest["Modalidade"] = "Mestrado"
doc["Modalidade"] = "Doutorado" 

# Remover a última linha do mestrado 
mest = mest.head(-1)
pos_grad = pd.concat([mest, doc], axis=0)
pos_grad = pos_grad.drop(columns="Matrícula")
pos_grad.sample(5)

,Aluno/a,Entrada,Área de Concentração,Orientador/a,Bolsista,Agência,Início bolsa,Término bolsa,Prazo término curso,Modalidade
29,João Victor Roriz Saboya,ago/23,Astrofísica,Gustavo Guerrero,Sim,CAPES,jan/24,dez/25,fev/26,Mestrado
48,Pablo Rodrigues de Souza Lima,mar/25,Física,Marcos Assunção Pimenta,Sim,CAPES,mar/25,fev/27,ago/27,Mestrado
37,João Alves De Oliveira Neto,jul/23,Física,Carlos Basílio,Sim,CAPES,jul/23,jun/27,jun/28,Doutorado
46,Noélia Yesenia Rojas Cruz,mar/24,Astrofísica,João Francisco Coelho,Sim,CNPq/PRPG,jun/24,ago/26,ago/26,Mestrado
5,Breno Martins Dos Anjos,out/21,Física,Raphael Drumond (Emmanuel Araújo – C),Não,NaN,NaN,NaN,mar/27,Doutorado


## Tratamento dos Dados

__Renomeando Colunas__


In [3]:
pos_grad.head(0)

,Aluno/a,Entrada,Área de Concentração,Orientador/a,Bolsista,Agência,Início bolsa,Término bolsa,Prazo término curso,Modalidade


In [4]:
# Renomeando colunas 
pos_grad.columns = pos_grad.columns.str.replace(" ", "_").str.lower()

# Removendo caracteres ( ´ , ç , ^)
cols_rename: dict[str, str] = {
    "matrícula": "matricula",
    "área_de_concentração": "area_de_concentracao",
    "agência": "agencia",
    "início_bolsa": "inicio_bolsa",
    "término_bolsa": "termino_bolsa",
    "prazo_término_curso": "prazo_termino_curso",
}
pos_grad = pos_grad.rename(columns=cols_rename)
pos_grad.head(0)

,aluno/a,entrada,area_de_concentracao,orientador/a,bolsista,agencia,inicio_bolsa,termino_bolsa,prazo_termino_curso,modalidade


__Tipos de dados__

In [5]:
pos_grad.info()

<class 'pandas.core.frame.DataFrame'>
Index: 152 entries, 0 to 82
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   aluno/a               152 non-null    object
 1   entrada               152 non-null    object
 2   area_de_concentracao  151 non-null    object
 3   orientador/a          152 non-null    object
 4   bolsista              152 non-null    object
 5   agencia               108 non-null    object
 6   inicio_bolsa          107 non-null    object
 7   termino_bolsa         107 non-null    object
 8   prazo_termino_curso   152 non-null    object
 9   modalidade            152 non-null    object
dtypes: object(10)
memory usage: 13.1+ KB


In [6]:
# Mudaremos colunas do tipo genérico object para tipos específicos.
# Format: mês/ano (jan/XX)
cols_datetime = ["entrada", "inicio_bolsa", "termino_bolsa", "prazo_termino_curso"] 
for col in cols_datetime:
    pos_grad[col] = pd.to_datetime(pos_grad[col], format="%b/%y")

In [7]:
# Convertendo bolsista para booleano
print("Valores únicos de bolsista (antes)", pos_grad["bolsista"].unique())
map_bolsista: dict[str, bool] = {"sim": True, "não": False}
pos_grad["bolsista"] = pos_grad["bolsista"].str.lower().map(map_bolsista)
print("Valores únicos de bolsista (depois)", pos_grad["bolsista"].unique())


Valores únicos de bolsista (antes) ['Não' 'Sim' 'SIm']
Valores únicos de bolsista (depois) [False  True]


__Formatação de Strings__

In [8]:
print(f"Agências de Fomento (antes): {pos_grad["agencia"].unique()}")

# Correção do typo CNPq/PRPG e padronização em maiúscula
pos_grad["agencia"] = pos_grad["agencia"].str.upper()

print(f"Agências de Fomento (depois): {pos_grad["agencia"].unique()}")


Agências de Fomento (antes): [nan 'CAPES' 'CNPq/PRPG' 'FAPEMIG' 'CNPQ' 'CAPES/PRPG' 'CNPQ/PRPG']
Agências de Fomento (depois): [nan 'CAPES' 'CNPQ/PRPG' 'FAPEMIG' 'CNPQ' 'CAPES/PRPG']


In [9]:
# Formatando nome de pessoas com "Nome Sobrenome Sobrenome2" 
for col in ["aluno/a", "orientador/a"]:
    pos_grad[col] = pos_grad[col].str.title().str.strip()


In [10]:
pos_grad

,aluno/a,entrada,area_de_concentracao,orientador/a,bolsista,agencia,inicio_bolsa,termino_bolsa,prazo_termino_curso,modalidade
0,Allan Gabriel Marques Lima,2022-04-01,Física,Mario Mazzoni,False,NaN,NaT,NaT,2025-09-01,Mestrado
1,Ana Carolina Dos Santos,2023-08-01,Física,Rogério Paniago,False,NaN,NaT,NaT,2026-01-01,Mestrado
2,André Felipe Alves Alencar,2023-08-01,Física,Mario Mazzoni,False,NaN,NaT,NaT,2026-01-01,Mestrado
3,Barbara Alves Land Ferreira,2023-03-01,Física,Simone Silva Alexandre,False,NaN,NaT,NaT,2026-03-01,Mestrado
4,Barbara Regina Melo Ribeiro,2024-03-01,Física,Ana Maria De Paula (Daniela Ferreira – C),True,CAPES,2024-03-01,2026-02-01,2026-08-01,Mestrado
...,...,...,...,...,...,...,...,...,...,...
78,Vinicius Aguiar Cardoso Godinho Baliza,2025-08-01,Física,Não Possui Orientador Cadastrado,False,NaN,NaT,NaT,2030-07-01,Doutorado
79,Vitor Assunção Moreira Lima,2022-06-01,Física,Walber Hugo De Brito,True,FAPEMIG,2022-07-01,2026-06-01,2027-05-01,Doutorado
80,Vitor Monteiro Macaroun,2025-08-01,Física,Não Possui Orientador Cadastrado,False,NaN,NaT,NaT,2030-07-01,Doutorado
81,Vitor Monteiro Pereira,2021-05-01,Física,Raphael Drumond,False,NaN,NaT,NaT,2026-11-01,Doutorado


In [11]:
# Armazenando a versão "limpa"
pos_grad.to_csv(path_or_buf="fisica-ufmg.csv", index=False)

---